# 카카오 로컬(Local) API 예시

주소 검색 · 지오코딩 · 역지오코딩 · 키워드 검색을 순서대로 실행해 봅니다.

**준비**
1. [카카오 개발자 콘솔](https://developers.kakao.com)에서 앱 생성 → **REST API 키** 발급
2. 프로젝트 루트에 `.env` 파일을 만들고 `KAKAO_REST_API_KEY=발급받은키` 입력

**공통 규칙**

| 항목 | 내용 |
| --- | --- |
| 좌표 순서 | `x` = 경도(lng), `y` = 위도(lat) — 헷갈리기 쉬우니 주의 |
| 인증 | 모든 요청 헤더에 `Authorization: KakaoAK <REST API 키>` |
| 응답 구조 | `{"meta": {...}, "documents": [...]}` — 실제 결과는 `documents` |
| 좌표 타입 | 응답의 `x`, `y`는 **문자열**입니다. 계산에 쓰려면 `float()` 변환 필요 |

**실행 순서**: 셀이 위에서 아래로 이어집니다. 2번에서 만든 `center` 변수를 3번 이후가 계속 쓰므로,
커널을 새로 시작했다면 0 → 1 → 2 순서로 실행하세요.

## 0. 준비

키를 읽고, 모든 요청에서 공통으로 쓸 주소(`BASE`)와 인증 헤더(`HEADERS`)를 만듭니다.

- `load_dotenv()` — `.env` 파일의 내용을 환경변수로 올립니다. 키를 코드에 직접 적지 않기 위함입니다.
- `HEADERS` — 카카오는 `KakaoAK ` 접두사가 붙은 REST API 키를 요구합니다. 형식이 틀리면 `401`이 납니다.
- 한 번 만들어 두면 이후 모든 셀에서 재사용합니다.

In [ ]:
# 최초 1회만 실행
# !pip install requests python-dotenv pandas

In [ ]:
import os
import time

import requests
from dotenv import load_dotenv

load_dotenv()                              # .env 에서 키를 읽는다
API_KEY = os.getenv("KAKAO_REST_API_KEY")

# .env 를 안 쓴다면 아래처럼 입력 (노트북에 키를 직접 적어 저장하지 말 것)
# import getpass
# API_KEY = getpass.getpass("REST API KEY: ")

BASE = "https://dapi.kakao.com/v2/local"
HEADERS = {"Authorization": f"KakaoAK {API_KEY}"}

print("키 설정됨:", bool(API_KEY))

## 1. 주소 검색

`/search/address.json` — **주소 문자열로 검색**합니다. 도로명·지번 어느 쪽을 넣어도 되고,
"판교역로 235"처럼 일부만 넣어도 유사한 후보들을 돌려줍니다.

**코드 설명**

- `params` 에 넣은 값들이 URL 뒤에 `?query=...&size=10` 형태로 붙습니다.
- `size` 는 한 번에 받을 개수(1~30), `page` 는 페이지 번호입니다.
- `res.json()` 이 응답 전체(`meta` + `documents`)이고, 여기서 `["documents"]` 만 꺼내 반환합니다.
- 결과 1건에는 `address`(지번)와 `road_address`(도로명)가 **각각 별도 객체**로 들어 있습니다.

In [ ]:
def search_address(query, size=10, page=1):
    """주소를 검색해 documents 리스트를 반환한다."""
    res = requests.get(
        f"{BASE}/search/address.json",
        headers=HEADERS,
        params={"query": query, "size": size, "page": page},
    )
    return res.json()["documents"]

In [ ]:
docs = search_address("판교역로 235")
print(f"{len(docs)}건")

for d in docs:
    road = d.get("road_address") or {}
    print(f"{d['address_name']} | {road.get('address_name', '-')} | {d['x']}, {d['y']}")

`road = d.get("road_address") or {}` 이 한 줄이 중요합니다.

도로명이 부여되지 않은 주소(산번지 등)는 `road_address` 값이 **`None`** 으로 옵니다.
`d.get("road_address", {})` 로 쓰면 키가 존재하므로 `None` 이 그대로 나와서
다음 줄에서 `None.get(...)` → `AttributeError` 가 납니다. `or {}` 는 `None` 도 빈 dict 로 바꿔줍니다.

In [ ]:
# 응답 구조가 궁금하면 원본을 그대로 확인
docs[0]

## 2. 지오코딩 (주소 → 좌표)

**지오코딩**은 사람이 읽는 주소를 좌표로 바꾸는 일입니다.
1번의 주소 검색 결과에서 필요한 값만 뽑아 쓰기 좋은 형태로 정리합니다.

**코드 설명**

- `size=1` — 가장 잘 맞는 후보 하나만 받습니다.
- 검색 결과가 없으면 `None` 을 반환합니다. 이후 셀에서 `center` 가 `None` 이면 에러가 나므로 값을 꼭 확인하세요.
- `float(d["x"])` — 응답이 문자열이라 숫자로 바꿔 둡니다.
- `region_1depth`(시도) / `2depth`(시군구) / `3depth`(읍면동)는 지역별 집계에 유용합니다.

In [ ]:
def geocode(address):
    """주소를 좌표로 변환한다. 결과가 없으면 None."""
    docs = search_address(address, size=1)
    if not docs:
        return None

    d = docs[0]
    road = d.get("road_address") or {}
    jibun = d.get("address") or {}
    return {
        "input": address,
        "lng": float(d["x"]),
        "lat": float(d["y"]),
        "road_address": road.get("address_name", ""),
        "jibun_address": jibun.get("address_name", ""),
        "zone_no": road.get("zone_no", ""),          # 우편번호
        "building": road.get("building_name", ""),
        "sido": jibun.get("region_1depth_name", ""),
        "sigungu": jibun.get("region_2depth_name", ""),
        "dong": jibun.get("region_3depth_name", ""),
    }

In [ ]:
center = geocode("경기도 성남시 분당구 판교역로 235")
center

여기서 만든 **`center` 가 이후 모든 섹션의 기준점**입니다.
3~8번 섹션이 `center["lng"]`, `center["lat"]` 를 중심 좌표로 씁니다.

다른 지역을 보고 싶으면 **위 셀의 주소만 바꾸면** 나머지가 전부 그 지점 기준으로 다시 돌아갑니다.
좌표를 이미 알고 있다면 `center = {"lng": 127.108, "lat": 37.401}` 처럼 직접 만들어도 됩니다.

### 2-1. 여러 주소를 한 번에

주소 목록을 반복문으로 돌립니다. 대량 처리 시 두 가지를 신경 써야 합니다.

- **실패 처리** — 검색이 안 되는 주소는 `None` 이 들어갑니다. 건너뛰지 말고 어떤 주소가 실패했는지 남기세요.
- **호출 간격** — `time.sleep(0.05)` 로 간격을 둡니다. 쿼터 초과(`429`)를 피하기 위함입니다.

In [ ]:
addresses = [
    "서울 중구 세종대로 110",          # 서울시청
    "서울 용산구 남산공원길 105",       # N서울타워
    "부산 해운대구 해운대해변로 264",
    "제주 제주시 첨단로 242",
    "없는주소 12345",                  # 실패 케이스
]

rows = []
for addr in addresses:
    rows.append(geocode(addr))
    time.sleep(0.05)                   # 호출 간격(쿼터 보호)

for addr, r in zip(addresses, rows):
    if r:
        print(f"{addr} -> ({r['lat']:.6f}, {r['lng']:.6f})  {r['building']}")
    else:
        print(f"{addr} -> 결과 없음")

## 3. 역지오코딩 (좌표 → 주소)

지오코딩의 반대입니다. 좌표를 넣으면 그 지점의 주소를 알려줍니다. 두 종류가 있습니다.

| 엔드포인트 | 돌려주는 것 | 쓰임새 |
| --- | --- | --- |
| `/geo/coord2address.json` | 도로명 / 지번 주소, 우편번호, 건물명 | 사람에게 보여줄 주소 |
| `/geo/coord2regioncode.json` | 행정동·법정동 이름과 **행정코드** | 통계·집계, 다른 공공데이터와 결합 |

**코드 설명**

- 파라미터가 `x`(경도), `y`(위도) 단 두 개입니다.
- `coord2regioncode` 는 보통 **2건**을 돌려줍니다. `region_type` 이 `B` 면 법정동, `H` 면 행정동입니다.
- `code` 는 10자리 행정구역 코드로, 다른 공공데이터와 조인할 때 씁니다.

In [ ]:
def reverse_geocode(lng, lat):
    """좌표를 주소로 변환한다."""
    res = requests.get(
        f"{BASE}/geo/coord2address.json",
        headers=HEADERS,
        params={"x": lng, "y": lat},
    )
    docs = res.json()["documents"]
    if not docs:
        return None

    road = docs[0].get("road_address") or {}
    jibun = docs[0].get("address") or {}
    return {
        "road_address": road.get("address_name", ""),
        "jibun_address": jibun.get("address_name", ""),
        "zone_no": road.get("zone_no", ""),
        "building": road.get("building_name", ""),
    }


def coord_to_region(lng, lat):
    """좌표의 행정동/법정동 정보를 반환한다."""
    res = requests.get(
        f"{BASE}/geo/coord2regioncode.json",
        headers=HEADERS,
        params={"x": lng, "y": lat},
    )
    return res.json()["documents"]

In [ ]:
print(reverse_geocode(center["lng"], center["lat"]))
print()

for r in coord_to_region(center["lng"], center["lat"]):
    # region_type: B=법정동, H=행정동
    print(f"[{r['region_type']}] {r['address_name']}  code={r['code']}")

## 4. 키워드 검색

`/search/keyword.json` — **장소명·업종명으로 검색**합니다. "스타벅스", "카페", "성남시청" 모두 됩니다.
중심 좌표와 반경을 주면 그 범위 안에서만 찾습니다.

**파라미터**

| 이름 | 설명 |
| --- | --- |
| `query` | 검색어 (필수) |
| `x`, `y` | 중심 좌표 (경도, 위도) |
| `radius` | 중심에서의 반경(m). 0~20000 |
| `page` / `size` | 페이지 번호(1~45) / 페이지당 개수(1~15) |
| `sort` | `accuracy`(기본, 정확도순) 또는 `distance`(거리순 — 좌표 필요) |
| `category_group_code` | 업종 코드로 추가 필터 (5번 섹션 참고) |

**코드 설명**

- 선택 파라미터의 기본값을 `None` 으로 두면, `requests` 가 **`None` 인 항목을 URL에서 알아서 뺍니다.**
  따로 걸러낼 필요가 없습니다.
- `return res.json()` — 여기서는 `documents` 만이 아니라 **응답 전체**를 돌려줍니다.
  `meta` 안의 `total_count`, `is_end` 를 써야 하기 때문입니다.
- 중심 좌표를 준 경우에만 결과에 `distance`(m, 문자열)가 채워집니다.

In [ ]:
def search_keyword(query, lng=None, lat=None, radius=None,
                   page=1, size=15, sort=None, category_group_code=None):
    """키워드로 장소를 검색한다. 원본 응답(meta + documents)을 반환."""
    params = {
        "query": query,
        "x": lng,
        "y": lat,
        "radius": radius,
        "page": page,
        "size": size,
        "sort": sort,
        "category_group_code": category_group_code,
    }
    res = requests.get(f"{BASE}/search/keyword.json", headers=HEADERS, params=params)
    return res.json()

In [ ]:
data = search_keyword(
    "카페",
    lng=center["lng"],
    lat=center["lat"],
    radius=1000,
    sort="distance",
    size=10,
)

print("전체:", data["meta"]["total_count"], "건 / 노출 가능:", data["meta"]["pageable_count"], "건")
print()

for d in data["documents"]:
    print(f"{d['distance']}m  {d['place_name']}  {d['road_address_name']}")

`meta` 의 두 숫자를 비교해 보세요.

- `total_count` — 조건에 맞는 **실제 전체 개수**
- `pageable_count` — 그중 **실제로 받아올 수 있는 개수** (최대 45)

`total_count` 가 `pageable_count` 보다 크면 **결과가 잘렸다**는 뜻입니다.
이 신호를 7번 섹션에서 격자 분할의 판단 근거로 씁니다.

### 4-1. 여러 페이지 모으기

한 번에 최대 15건이므로, 더 받으려면 `page` 를 넘겨가며 여러 번 호출해 합쳐야 합니다.

**코드 설명**

- `**kwargs` — 여기 적지 않은 나머지 인자(`lng`, `lat`, `radius`, `sort` …)를 dict 로 받아
  아래에서 `search_keyword(...)` 에 그대로 넘깁니다. 파라미터가 늘어나도 이 함수는 고칠 필요가 없습니다.
- `kwargs.pop("page", None)` — 호출자가 실수로 `page` 를 넘기면 아래에서 `page` 가 두 번 전달되어
  `TypeError` 가 납니다. 페이지 관리는 이 함수의 몫이므로 미리 제거합니다.
- `extend` vs `append` — `extend` 는 요소를 펼쳐 이어붙이고, `append` 는 리스트를 통째로 넣습니다.
  평평한 리스트가 필요하므로 `extend` 입니다.
- `is_end` — 카카오가 "마지막 페이지"라고 알려주는 값입니다. 이걸 무시하면 빈 요청으로 쿼터만 씁니다.
- `time.sleep` 이 `break` **뒤에** 있어서, 마지막 반복에서는 불필요하게 쉬지 않습니다.

**한계**: 이 함수로도 **45건을 넘길 수 없습니다.** 그건 7번 섹션에서 해결합니다.

In [ ]:
def search_keyword_all(query, max_results=45, **kwargs):
    """is_end 가 될 때까지 페이지를 넘기며 모은다 (최대 45건)."""
    kwargs.pop("page", None)
    kwargs.pop("size", None)

    results = []
    for page in range(1, 46):
        data = search_keyword(query, page=page, size=15, **kwargs)
        results.extend(data["documents"])

        if data["meta"]["is_end"] or len(results) >= max_results:
            break
        time.sleep(0.05)

    return results[:max_results]

In [ ]:
places = search_keyword_all(
    "카페", lng=center["lng"], lat=center["lat"], radius=1000, sort="distance"
)
print(len(places), "건 수집")

## 5. 카테고리 검색

`/search/category.json` — **업종 코드로** 주변을 훑습니다.
"지하철역"처럼 키워드로 검색하면 상호에 그 단어가 든 엉뚱한 가게가 섞이는데,
카테고리 코드를 쓰면 업종이 정확히 일치하는 것만 나옵니다. 입지 분석에는 이쪽이 적합합니다.

**코드 설명**

- `category_group_code` 가 필수이고, 반경 검색을 하려면 `x`, `y` 도 필요합니다.
- `sort="distance"` 로 가까운 순 정렬합니다.
- 키워드 검색과 마찬가지로 **최대 45건** 제한이 적용됩니다.

In [ ]:
CATEGORY_GROUP = {
    "대형마트": "MT1", "편의점": "CS2", "어린이집·유치원": "PS3",
    "학교": "SC4", "학원": "AC5", "주차장": "PK6",
    "주유소·충전소": "OL7", "지하철역": "SW8", "은행": "BK9",
    "문화시설": "CT1", "중개업소": "AG2", "공공기관": "PO3",
    "관광명소": "AT4", "숙박": "AD5", "음식점": "FD6",
    "카페": "CE7", "병원": "HP8", "약국": "PM9",
}


def search_category(code, lng, lat, radius=1000, page=1, size=15, sort="distance"):
    """카테고리 코드로 주변 장소를 검색한다."""
    res = requests.get(
        f"{BASE}/search/category.json",
        headers=HEADERS,
        params={
            "category_group_code": code,
            "x": lng,
            "y": lat,
            "radius": radius,
            "page": page,
            "size": size,
            "sort": sort,
        },
    )
    return res.json()["documents"]

In [ ]:
for name in ["지하철역", "학교", "대형마트"]:
    found = search_category(
        CATEGORY_GROUP[name], center["lng"], center["lat"], radius=1500
    )
    print(f"--- {name} ({len(found)}건) ---")
    for d in found[:5]:
        print(f"  {d['distance']}m  {d['place_name']}")
    print()

## 6. 결과를 표로 정리

수집한 결과를 `pandas.DataFrame` 으로 옮기면 필터링·집계·저장이 쉬워집니다.

**코드 설명**

- `pd.DataFrame(places)` — dict 리스트를 그대로 넣으면 키가 컬럼이 됩니다.
- `[[...]]` — 필요한 컬럼만 골라냅니다. 카카오 응답에는 안 쓰는 필드가 많습니다.
- `rename(columns={"x": "lng", "y": "lat"})` — `x`/`y` 는 의미가 불분명하니 이름을 바꿔 둡니다.
- `astype(int)` / `astype(float)` — 응답이 전부 문자열이라, 비교·정렬하려면 숫자로 바꿔야 합니다.
  이걸 빠뜨리면 `"1000" < "500"` 같은 문자열 비교가 되어 결과가 틀립니다.
- `encoding="utf-8-sig"` — 엑셀에서 CSV를 열 때 한글이 깨지지 않게 하는 설정입니다.

In [ ]:
import pandas as pd

df = pd.DataFrame(places)[
    ["place_name", "category_name", "road_address_name", "phone", "place_url", "distance", "x", "y"]
]
df = df.rename(columns={"x": "lng", "y": "lat"})
df["distance"] = df["distance"].astype(int)
df[["lng", "lat"]] = df[["lng", "lat"]].astype(float)

df.head(10)

In [ ]:
# 예: 500m 이내만, 카테고리 소분류별 개수
near = df[df["distance"] <= 500]
print(f"500m 이내 {len(near)}건")

near["category_name"].str.split(" > ").str[-1].value_counts()

In [ ]:
# CSV 로 저장
df.to_csv("kakao_places.csv", index=False, encoding="utf-8-sig")
print("저장 완료: kakao_places.csv")

## 7. 지도에 표기

4-1절에서 모은 **45건(`places` / `df`)** 을 지도에 올립니다. 두 가지 방식을 씁니다.

| 방식 | 라이브러리 | 특징 |
| --- | --- | --- |
| **정적** | matplotlib | 이미지(PNG). 배경 지도는 없지만 보고서·문서에 넣기 좋음 |
| **동적** | folium | 브라우저에서 확대·이동·클릭 가능. 실제 지도 배경 위에 표시 |

둘 다 좌표 처리에서 같은 함정이 있습니다 — **위도 1도와 경도 1도의 실제 길이가 다릅니다.**
위도 1도는 어디서나 약 111km지만, 경도 1도는 위도 37도에서 약 88km입니다.
이걸 무시하면 지도가 가로로 눌려 보이고 반경 원도 어긋납니다.

### 7-1. 정적 지도 (matplotlib)

점 하나가 장소 하나이고, **색으로 중심에서의 거리**를 나타냅니다.

**코드 설명**

- `scatter(..., c=df["distance"], cmap="viridis_r")` — `c` 에 숫자 컬럼을 주면 값에 따라 색이 정해집니다.
  `_r` 은 색상표를 뒤집는다는 뜻으로, 가까울수록 밝게 만듭니다.
- `set_aspect(1 / cos(위도))` — 위 설명의 길이 차이를 보정합니다. 이게 없으면 원이 타원으로 보입니다.
- **반경 원은 `Circle` 이 아니라 `Ellipse`** 로 그립니다. 데이터 좌표가 degree 단위인데
  가로(경도)와 세로(위도)의 degree당 미터가 다르기 때문입니다. `Circle` 을 쓰면
  실제보다 좁은 범위가 그려져 원 밖에 점이 찍히는 오류가 납니다.
- `nsmallest(5, "distance")` — 가까운 5곳만 이름을 답니다. 45개 전부 달면 글자가 겹쳐 못 읽습니다.
- 한글이 네모로 깨지면 폰트 설정 부분을 확인하세요 (macOS는 `AppleGothic`, Windows는 `Malgun Gothic`).

In [ ]:
import math

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

M_PER_DEG_LAT = 111_320.0

# 한글 폰트 (설치된 것 중 처음 발견되는 것을 사용)
for _cand in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f.name == _cand for f in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = _cand
        break
plt.rcParams["axes.unicode_minus"] = False   # 마이너스 기호 깨짐 방지

print("사용 폰트:", plt.rcParams["font.family"])

In [ ]:
def plot_static(df, center, radius=1000, label_top=5, title="검색 결과"):
    """검색 결과를 정적 지도(산점도)로 그린다."""
    fig, ax = plt.subplots(figsize=(8, 8))

    sc = ax.scatter(df["lng"], df["lat"], c=df["distance"], cmap="viridis_r",
                    s=45, alpha=0.85, edgecolors="white", linewidths=0.5, zorder=3)
    ax.scatter([center["lng"]], [center["lat"]], marker="*", s=400,
               color="crimson", edgecolors="white", linewidths=1, zorder=5, label="중심")

    deg_lat = radius / M_PER_DEG_LAT
    deg_lng = radius / (M_PER_DEG_LAT * math.cos(math.radians(center["lat"])))

    # 위도/경도의 1도 길이가 달라, 실제 원을 그리려면 가로세로 반경을 따로 준다
    ax.add_patch(Ellipse((center["lng"], center["lat"]),
                         width=2 * deg_lng, height=2 * deg_lat,
                         fill=False, color="crimson", ls="--", lw=1.2, zorder=2))

    for _, r in df.nsmallest(label_top, "distance").iterrows():
        ax.annotate(r["place_name"], (r["lng"], r["lat"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=9, zorder=6)

    ax.set_aspect(1 / math.cos(math.radians(center["lat"])))
    ax.set_xlim(center["lng"] - deg_lng * 1.15, center["lng"] + deg_lng * 1.15)
    ax.set_ylim(center["lat"] - deg_lat * 1.15, center["lat"] + deg_lat * 1.15)

    fig.colorbar(sc, ax=ax, label="중심에서의 거리 (m)", shrink=0.8)
    ax.set_title(f"{title} ({len(df)}건)")
    ax.set_xlabel("경도")
    ax.set_ylabel("위도")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
    return fig, ax

In [ ]:
fig, ax = plot_static(df, center, radius=1000, title="반경 1km 카페")
fig.savefig("kakao_static_map.png", dpi=110, bbox_inches="tight")
plt.show()

### 7-2. 동적 지도 (folium)

실제 지도 배경 위에 마커를 찍습니다. 마커에 **마우스를 올리면 이름**, **클릭하면 상세 정보**가 뜹니다.

**코드 설명**

- `folium.Map(location=[위도, 경도])` — **folium은 `[lat, lng]` 순서**입니다.
  카카오의 `x, y`(경도, 위도)와 **순서가 반대**라 여기서 실수가 잦습니다.
- `folium.Circle(radius=1000)` — folium의 `radius` 는 **미터 단위**라 좌표 변환이 필요 없습니다.
  (반면 `CircleMarker` 의 `radius` 는 픽셀 단위입니다. 이름이 비슷하지만 다릅니다.)
- `tooltip` 은 마우스를 올렸을 때, `popup` 은 클릭했을 때 나옵니다.
- `place_url` 은 카카오가 주는 장소 상세 페이지 링크로, 팝업에서 바로 열 수 있게 넣었습니다.
- 노트북에서는 마지막 줄에 `m` 을 두면 그대로 렌더링되고, `m.save("...html")` 로 파일로도 남길 수 있습니다.

In [ ]:
import folium


def make_map(df, center, radius=1000, zoom=15):
    """검색 결과를 folium 동적 지도로 만든다."""
    m = folium.Map(location=[center["lat"], center["lng"]], zoom_start=zoom,
                   tiles="OpenStreetMap")

    # 검색 반경 (folium 의 Circle radius 는 미터 단위)
    folium.Circle([center["lat"], center["lng"]], radius=radius,
                  color="crimson", fill=True, fill_opacity=0.06, weight=2).add_to(m)

    folium.Marker([center["lat"], center["lng"]], tooltip="중심",
                  icon=folium.Icon(color="red", icon="star")).add_to(m)

    for _, r in df.iterrows():
        html = (
            f"<b>{r['place_name']}</b><br>"
            f"{r['road_address_name']}<br>"
            f"{r['distance']}m · {r['phone'] or '-'}<br>"
            f"<a href='{r.get('place_url', '')}' target='_blank'>카카오맵에서 보기</a>"
        )
        folium.CircleMarker(
            [r["lat"], r["lng"]], radius=6,                # 여기 radius 는 픽셀
            color="#1f77b4", fill=True, fill_opacity=0.8, weight=1,
            tooltip=r["place_name"],
            popup=folium.Popup(html, max_width=260),
        ).add_to(m)

    return m

In [ ]:
m = make_map(df, center, radius=1000)
m.save("kakao_dynamic_map.html")
m

팝업의 "카카오맵에서 보기" 링크는 6번 섹션에서 `place_url` 컬럼을 포함시켰기 때문에 동작합니다.
`KeyError: 'place_url'` 이 난다면 6번 셀의 컬럼 목록을 확인하세요.

**HTML 파일로 저장**하면 노트북 없이도 브라우저에서 열어볼 수 있고, 그대로 공유할 수 있습니다.
지도 타일은 열 때 인터넷에서 받아오므로 오프라인에서는 배경이 비어 보입니다.

### 7-3. 여러 카테고리를 한 지도에 겹치기

"편의점 45건 + 카페 45건 + 주차장 45건"처럼 **업종별로 따로 모아 한 지도에 색을 나눠** 올립니다.
45건 상한은 **조건마다 따로** 적용되므로, 3개 업종이면 최대 135건까지 모을 수 있습니다.

**코드 설명**

- `collect_categories()` — 업종 이름 목록을 받아 각각 `search_keyword_all()` 로 45건까지 모으고,
  **`group` 컬럼**(업종 이름)을 붙여 하나의 DataFrame 으로 합칩니다. 이 컬럼이 색을 나누는 기준이 됩니다.
- `category_group_code` 를 함께 넘겨서 상호에 그 단어가 든 엉뚱한 가게가 섞이지 않게 합니다.
- **정적 지도**: `df.groupby("group")` 으로 나눠 `scatter` 를 여러 번 호출합니다.
  한 번에 다 그리면 색을 나눌 수 없고, 범례도 안 생깁니다.
- **동적 지도**: 업종마다 `folium.FeatureGroup` 을 만들고 마지막에 `LayerControl` 을 붙입니다.
  그러면 지도 우측 상단에 **체크박스**가 생겨 업종별로 켜고 끌 수 있습니다.

In [ ]:
GROUP_COLORS = {
    "카페": "#e6550d", "편의점": "#3182bd", "주차장": "#31a354",
    "지하철역": "#756bb1", "학교": "#e377c2", "은행": "#8c564b",
    "대형마트": "#17becf", "병원": "#d62728", "약국": "#7f7f7f",
}


def collect_categories(names, center, radius=1000):
    """업종 목록을 각각 45건까지 모아 group 컬럼을 붙여 합친다."""
    frames = []

    for name in names:
        docs = search_keyword_all(
            name,
            lng=center["lng"], lat=center["lat"], radius=radius, sort="distance",
            category_group_code=CATEGORY_GROUP.get(name),
        )
        print(f"{name}: {len(docs)}건")
        if not docs:
            continue

        g = pd.DataFrame(docs)[
            ["place_name", "road_address_name", "phone", "place_url", "distance", "x", "y"]
        ].rename(columns={"x": "lng", "y": "lat"})
        g.insert(0, "group", name)
        frames.append(g)

    out = pd.concat(frames, ignore_index=True)
    out["distance"] = out["distance"].astype(int)
    out[["lng", "lat"]] = out[["lng", "lat"]].astype(float)
    return out

In [ ]:
multi_df = collect_categories(["카페", "편의점", "주차장"], center, radius=1000)

print()
print("합계", len(multi_df), "건")
multi_df["group"].value_counts()

In [ ]:
def plot_static_multi(df, center, radius=1000, title="카테고리별 분포"):
    """업종별로 색을 나눠 정적 지도를 그린다."""
    fig, ax = plt.subplots(figsize=(8.5, 8))

    for name, g in df.groupby("group", sort=False):
        ax.scatter(g["lng"], g["lat"], s=42, alpha=0.8,
                   color=GROUP_COLORS.get(name, "#999999"),
                   edgecolors="white", linewidths=0.5, zorder=3,
                   label=f"{name} ({len(g)})")

    ax.scatter([center["lng"]], [center["lat"]], marker="*", s=400, color="black",
               edgecolors="white", linewidths=1, zorder=5, label="중심")

    deg_lat = radius / M_PER_DEG_LAT
    deg_lng = radius / (M_PER_DEG_LAT * math.cos(math.radians(center["lat"])))
    ax.add_patch(Ellipse((center["lng"], center["lat"]),
                         width=2 * deg_lng, height=2 * deg_lat,
                         fill=False, color="black", ls="--", lw=1, alpha=0.5, zorder=2))

    ax.set_aspect(1 / math.cos(math.radians(center["lat"])))
    ax.set_xlim(center["lng"] - deg_lng * 1.15, center["lng"] + deg_lng * 1.15)
    ax.set_ylim(center["lat"] - deg_lat * 1.15, center["lat"] + deg_lat * 1.15)

    ax.set_title(f"{title} (총 {len(df)}건)")
    ax.set_xlabel("경도")
    ax.set_ylabel("위도")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", framealpha=0.9)
    return fig, ax

In [ ]:
fig, ax = plot_static_multi(multi_df, center, radius=1000)
fig.savefig("kakao_static_map_multi.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
def make_map_multi(df, center, radius=1000, zoom=15):
    """업종별 레이어로 나눈 동적 지도. 우측 상단에서 켜고 끌 수 있다."""
    m = folium.Map(location=[center["lat"], center["lng"]], zoom_start=zoom,
                   tiles="OpenStreetMap")

    folium.Circle([center["lat"], center["lng"]], radius=radius,
                  color="black", fill=False, weight=1, dash_array="5").add_to(m)
    folium.Marker([center["lat"], center["lng"]], tooltip="중심",
                  icon=folium.Icon(color="red", icon="star")).add_to(m)

    for name, g in df.groupby("group", sort=False):
        color = GROUP_COLORS.get(name, "#999999")
        fg = folium.FeatureGroup(name=f"{name} ({len(g)})", show=True)

        for _, r in g.iterrows():
            popup = (f"<b>{r['place_name']}</b><br>{r['group']}<br>"
                     f"{r['road_address_name']}<br>{r['distance']}m")
            folium.CircleMarker(
                [r["lat"], r["lng"]], radius=6, color=color,
                fill=True, fill_color=color, fill_opacity=0.85, weight=1,
                tooltip=f"[{r['group']}] {r['place_name']}",
                popup=folium.Popup(popup, max_width=240),
            ).add_to(fg)

        fg.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)   # 업종별 체크박스
    return m

In [ ]:
m2 = make_map_multi(multi_df, center, radius=1000)
m2.save("kakao_dynamic_map_multi.html")
m2

**응용**

- 업종별 개수만 비교하려면 `multi_df["group"].value_counts()` 로 충분합니다.
- 특정 업종의 최근접 거리는 `multi_df.groupby("group")["distance"].min()` 으로 구합니다.
  "가장 가까운 편의점까지 몇 m" 같은 입지 지표가 됩니다.
- 업종을 늘리려면 `collect_categories(["카페", "편의점", "주차장", "은행", "병원"], ...)` 처럼
  이름만 추가하면 됩니다. `CATEGORY_GROUP` 에 있는 이름은 코드 필터까지 자동 적용됩니다.
- 업종 수만큼 호출이 늘어납니다(업종당 최대 3회). 개수가 많으면 쿼터를 신경 쓰세요.

### 7-4. (선택) 카카오맵 위에 직접 표기

배경을 **카카오맵**으로 쓰고 싶다면 JavaScript SDK를 노트북에 끼워 넣는 방법이 있습니다.
다만 준비물이 다릅니다.

- **JavaScript 키**가 따로 필요합니다 (지금까지 쓴 REST API 키와 **다른 키**입니다).
- 카카오 개발자 콘솔 > 내 애플리케이션 > **플랫폼 > Web** 에 노트북 주소
  (예: `http://localhost:8888`)를 **등록**해야 지도가 뜹니다. 등록하지 않으면 빈 화면만 나옵니다.

> 지도가 안 뜨면 브라우저 개발자도구 콘솔의 오류 메시지를 먼저 확인하세요.
> 편집기·환경에 따라 노트북 출력 안에서 지도가 뜨지 않는 경우도 있습니다.
> 그럴 때는 7-5절처럼 HTML 파일로 뽑아 여는 방법이 있습니다.
> 배경 지도가 꼭 카카오맵이어야 하는 게 아니라면 7-2의 folium 이 훨씬 간단합니다.

In [ ]:
import html as html_mod
import json

from IPython.display import HTML

KAKAO_JS_KEY = os.getenv("KAKAO_JS_KEY", "")   # .env 에 KAKAO_JS_KEY 추가 필요


def kakao_map(df, center, height=520, level=5):
    """카카오맵 JS SDK 를 iframe 으로 띄운다."""
    points = json.dumps(
        [{"lat": float(r["lat"]), "lng": float(r["lng"]), "name": str(r["place_name"])}
         for _, r in df.iterrows()],
        ensure_ascii=False,
    )

    doc = f"""
<!DOCTYPE html><html><head><meta charset="utf-8"></head>
<body style="margin:0">
<div id="map" style="width:100%;height:{height}px"></div>
<script src="https://dapi.kakao.com/v2/maps/sdk.js?appkey={KAKAO_JS_KEY}"></script>
<script>
  var center = new kakao.maps.LatLng({center["lat"]}, {center["lng"]});
  var map = new kakao.maps.Map(document.getElementById('map'), {{center: center, level: {level}}});
  var info = new kakao.maps.InfoWindow({{zIndex: 1}});

  {points}.forEach(function (p) {{
    var marker = new kakao.maps.Marker({{
      map: map, position: new kakao.maps.LatLng(p.lat, p.lng), title: p.name
    }});
    kakao.maps.event.addListener(marker, 'click', function () {{
      info.setContent('<div style="padding:5px;font-size:12px">' + p.name + '</div>');
      info.open(map, marker);
    }});
  }});
</script>
</body></html>
"""
    return HTML(
        f'<iframe srcdoc="{html_mod.escape(doc, quote=True)}" '
        f'style="width:100%;height:{height}px;border:0"></iframe>'
    )

In [ ]:
if KAKAO_JS_KEY:
    display(kakao_map(df, center))
else:
    print("KAKAO_JS_KEY 가 없습니다. .env 에 JavaScript 키를 추가하세요.")

### 7-5. 카카오맵 — HTML 파일로 저장해 브라우저에서 보기

7-4절의 지도는 노트북 안에서만 볼 수 있습니다. 이 절은 같은 지도를 **독립 HTML 파일**로 뽑습니다.

- 노트북 없이 **브라우저에서 바로** 열 수 있습니다.
- 파일 하나만 전달하면 되므로 **공유하기 쉽습니다.**
- 편집기·환경에 따라 7-4절이 노트북 출력에서 안 뜰 때의 **대안**이 됩니다.

주의할 점은 **출처(origin)가 노트북과 달라진다**는 것입니다. 아래처럼 로컬 서버로 열면
출처가 `http://localhost:8080` 이 되므로, 그 주소를 콘솔에 **따로 등록**해야 합니다.

**실행 순서**

1. 아래 셀로 `kakao_map.html` 을 만듭니다.
2. 터미널에서 그 폴더를 서빙합니다.
   ```bash
   python -m http.server 8080
   ```
3. 브라우저에서 `http://localhost:8080/kakao_map.html` 을 엽니다.
4. 카카오 콘솔 > **JavaScript SDK 도메인**에 `http://localhost:8080` 을 등록합니다.

**파일을 더블클릭해서 여는 방식(`file://`)은 안 됩니다.** `file://` 도 등록할 수 없는 형식이라
같은 문제가 생깁니다. 반드시 로컬 서버를 거쳐야 합니다.

**코드 설명**

- `json.dumps(..., ensure_ascii=False)` — 파이썬 리스트를 그대로 JS 배열 리터럴로 넣습니다.
  한글을 유니코드 이스케이프하지 않아 파일을 열어봐도 읽을 수 있습니다.
- `points.replace("</", "<\\/")` — 장소명에 `</script>` 가 들어 있으면 스크립트 블록이
  중간에 끊깁니다. JS 문자열 안에서 `<\/` 는 `</` 와 같은 의미라 안전하게 무력화됩니다.
- **`CustomOverlay`** — 기본 `Marker` 는 핀 이미지라 업종별 색을 나눌 수 없습니다.
  색칠한 `div` 를 직접 올려서 `GROUP_COLORS` 를 그대로 적용합니다.
- `InfoWindow` 를 마커마다 만들지 않고 **하나만 만들어 재사용**합니다.
  각각 만들면 클릭할 때마다 창이 쌓입니다.
- `group` 컬럼이 있으면 범례가 자동으로 붙습니다 (7-3절의 `multi_df` 를 그대로 넣어도 됩니다).

In [ ]:
import json


def save_kakao_map_html(df, center, path="kakao_map.html", level=5, radius=1000,
                        title="카카오맵 검색 결과"):
    """카카오맵 SDK 로 마커를 찍는 독립 HTML 파일을 저장한다."""
    if not KAKAO_JS_KEY:
        raise ValueError("KAKAO_JS_KEY 가 없습니다. .env 에 JavaScript 키를 추가하세요.")

    has_group = "group" in df.columns

    points = json.dumps(
        [
            {
                "lat": float(r["lat"]),
                "lng": float(r["lng"]),
                "name": str(r["place_name"]),
                "addr": str(r.get("road_address_name", "")),
                "dist": int(r["distance"]),
                "group": str(r["group"]) if has_group else "",
                "color": GROUP_COLORS.get(r["group"], "#1f77b4") if has_group else "#1f77b4",
            }
            for _, r in df.iterrows()
        ],
        ensure_ascii=False,
    )
    points = points.replace("</", "<\\/")   # 장소명에 </script> 가 들어가도 안전하게

    if has_group:
        legend = "".join(
            f'<div><span class="key" style="background:{GROUP_COLORS.get(g, "#1f77b4")}"></span>'
            f'{g} ({c})</div>'
            for g, c in df["group"].value_counts().items()
        )
    else:
        legend = f'<div><span class="key" style="background:#1f77b4"></span>결과 ({len(df)})</div>'

    doc = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="utf-8">
<title>{title}</title>
<style>
  html, body {{ margin: 0; height: 100%; font-family: sans-serif; }}
  #map {{ width: 100%; height: 100%; }}
  #legend {{ position: absolute; top: 12px; right: 12px; z-index: 5;
             background: rgba(255,255,255,.94); padding: 10px 14px;
             border-radius: 6px; font-size: 13px; box-shadow: 0 1px 6px rgba(0,0,0,.25); }}
  #legend div {{ margin: 3px 0; }}
  .key {{ display: inline-block; width: 11px; height: 11px; border-radius: 50%;
          margin-right: 7px; vertical-align: middle; }}
  .dot {{ width: 13px; height: 13px; border-radius: 50%; border: 2px solid #fff;
          box-shadow: 0 0 3px rgba(0,0,0,.5); cursor: pointer; }}
</style>
</head>
<body>
<div id="map"></div>
<div id="legend">{legend}</div>

<script src="https://dapi.kakao.com/v2/maps/sdk.js?appkey={KAKAO_JS_KEY}"></script>
<script>
  var center = new kakao.maps.LatLng({center["lat"]}, {center["lng"]});
  var map = new kakao.maps.Map(document.getElementById('map'),
                               {{ center: center, level: {level} }});

  new kakao.maps.Circle({{
    map: map, center: center, radius: {radius},
    strokeWeight: 2, strokeColor: '#d62728', strokeStyle: 'dashed',
    fillColor: '#d62728', fillOpacity: 0.03
  }});

  new kakao.maps.Marker({{ map: map, position: center, title: '중심' }});

  var info = new kakao.maps.InfoWindow({{ zIndex: 1 }});   // 하나만 만들어 재사용

  {points}.forEach(function (p) {{
    var pos = new kakao.maps.LatLng(p.lat, p.lng);
    var el = document.createElement('div');
    el.className = 'dot';
    el.style.background = p.color;
    el.title = p.name;
    el.onclick = function () {{
      info.setContent(
        '<div style="padding:6px 9px;font-size:12px;white-space:nowrap">' +
        '<b>' + p.name + '</b>' + (p.group ? ' (' + p.group + ')' : '') + '<br>' +
        p.addr + '<br>' + p.dist + 'm</div>'
      );
      info.setPosition(pos);
      info.open(map);
    }};
    new kakao.maps.CustomOverlay({{
      map: map, position: pos, content: el, xAnchor: 0.5, yAnchor: 0.5
    }});
  }});
</script>
</body>
</html>
"""

    with open(path, "w", encoding="utf-8") as f:
        f.write(doc)

    folder = os.path.dirname(os.path.abspath(path)) or "."
    print("저장 완료:", os.path.abspath(path))
    print()
    print("터미널에서:")
    print(f"  cd {folder}")
    print("  python -m http.server 8080")
    print()
    print(f"브라우저에서: http://localhost:8080/{os.path.basename(path)}")
    print("(카카오 콘솔 > JavaScript SDK 도메인에 http://localhost:8080 등록 필요)")
    return os.path.abspath(path)

In [ ]:
# 단일 검색 결과
save_kakao_map_html(df, center, path="kakao_map.html", radius=1000)

In [ ]:
# 업종별 결과 (7-3절의 multi_df) - 범례가 자동으로 붙는다
save_kakao_map_html(multi_df, center, path="kakao_map_multi.html", radius=1000)

**지도가 안 뜬다면** 브라우저 개발자도구(F12) > Console 탭을 먼저 확인하세요.

| 증상 | 원인 |
| --- | --- |
| 흰 화면, 콘솔에 도메인 관련 오류 | JavaScript SDK 도메인 미등록 — 포트까지 정확히 일치해야 함 |
| `appkey` 오류 | REST API 키를 넣었을 가능성 — **JavaScript 키**를 써야 함 |
| 지도는 뜨는데 마커가 없음 | `df` 가 비었거나 `lat`/`lng` 가 문자열 — `float` 변환 확인 |
| 주소창이 `file://` | 로컬 서버 없이 파일을 직접 연 경우 — `http.server` 로 열어야 함 |

`http.server` 는 터미널에서 `Ctrl+C` 로 끕니다. 8080 포트가 이미 쓰이고 있으면
`python -m http.server 8090` 처럼 바꾸고, 콘솔에도 그 포트를 등록하세요.

## 8. 격자 분할 — 45건 상한 넘기기

키워드·카테고리 검색은 조건당 최대 45건입니다. 반경 안의 장소를 **전부** 모으려면
영역을 쪼개서 여러 번 검색한 뒤 합쳐야 합니다.

**왜 균등 격자가 아닌가**

처음부터 100m 격자로 촘촘히 나누면 텅 빈 지역까지 훑어서 호출이 낭비됩니다.
여기서는 **상한에 걸린 셀만 4등분해서 다시 들어가는** 재귀 분할을 씁니다.
번화가는 잘게 쪼개지고, 한적한 곳은 한 번에 끝납니다.

**상한 판별**

매직넘버 45 대신 4번 섹션에서 본 메타값을 씁니다 — `total_count > pageable_count` 이면 잘린 것입니다.

**전체 흐름**

```
원 (중심, 반경 1000m)
  └ 45건 넘게 잘렸나?
       ├ 아니오 → 결과 담고 종료
       └ 예     → 4개 원으로 쪼개서 각각 다시 검사 (반경이 min_radius 이하가 될 때까지)
최종: id 로 중복 제거 → 원래 반경 밖 결과 필터 → 거리순 정렬
```

### 8-1. 좌표 유틸

미터를 위경도 차이로 바꾸는 함수와, 두 좌표 사이 거리를 재는 함수입니다.

- 위도 1도는 어디서나 약 **111,320m** 입니다.
- 경도 1도의 실제 거리는 위도에 따라 줄어듭니다(극지방으로 갈수록 좁아짐).
  그래서 `cos(위도)` 를 곱해 보정합니다.
- 지구를 평면으로 근사하므로 수 km 범위에서만 정확합니다. 이 용도에는 충분합니다.

In [ ]:
import math

M_PER_DEG_LAT = 111_320.0


def move(lng, lat, dx_m, dy_m):
    """중심에서 동쪽 dx_m, 북쪽 dy_m 이동한 좌표."""
    dlat = dy_m / M_PER_DEG_LAT
    dlng = dx_m / (M_PER_DEG_LAT * math.cos(math.radians(lat)))
    return lng + dlng, lat + dlat


def distance_m(lng1, lat1, lng2, lat2):
    """두 좌표 사이 직선거리(m). 짧은 거리용 근사."""
    dy = (lat2 - lat1) * M_PER_DEG_LAT
    dx = (lng2 - lng1) * M_PER_DEG_LAT * math.cos(math.radians((lat1 + lat2) / 2))
    return math.hypot(dx, dy)

### 8-2. 셀 하나 긁기

원 하나에서 받아올 수 있는 만큼(최대 45건) 받고, **결과가 잘렸는지 여부를 함께 반환**합니다.
이 두 번째 반환값이 "더 쪼갤지" 판단하는 근거가 됩니다.

- `range(1, 4)` — `size=15` 로 3페이지면 45건이라 그 이상은 볼 필요가 없습니다.
- `meta` 를 루프 밖 변수로 둔 이유는, 반복이 끝난 뒤 마지막 응답의 메타를 봐야 하기 때문입니다.
- 반환값이 `(문서들, 잘렸는지)` 형태의 튜플입니다.

In [ ]:
def fetch_cell(query, lng, lat, radius, **kwargs):
    """셀 하나를 최대치까지 긁는다. (문서들, 상한에 걸렸는지) 반환."""
    docs, meta = [], {}

    for page in range(1, 4):                  # size 15 x 3 = 45
        data = search_keyword(query, lng=lng, lat=lat, radius=int(radius),
                              page=page, size=15, **kwargs)
        meta = data["meta"]
        docs.extend(data["documents"])
        if meta["is_end"]:
            break
        time.sleep(0.05)

    truncated = meta["total_count"] > meta["pageable_count"]
    return docs, truncated

### 8-3. 재귀 분할 수집

잘린 셀을 4등분해 다시 넣는 과정을 반복합니다.

- **`stack`** — 아직 처리하지 않은 원들의 목록입니다. 함수 재귀는 깊이 제한에 걸릴 수 있어
  `while stack` 반복문으로 만들었습니다.
- **`found[d["id"]] = d`** — 분할한 원들이 서로 겹쳐서 같은 장소가 여러 번 잡힙니다.
  카카오가 주는 장소 `id` 를 키로 쓰면 자동으로 중복이 제거됩니다.
- **`sub_r = r * 0.7072`** — 원을 감싼 정사각형을 4등분하면 각 사분면은 한 변이 `r` 인 정사각형입니다.
  그걸 덮으려면 반지름이 대각선의 절반, 즉 `r × √2/2 ≈ 0.7071` 이어야 합니다. 약간 여유를 뒀습니다.
- **`min_radius`** — 이보다 작아지면 더 쪼개지 않고 포기합니다(`give_up` 으로 셉니다).
  무한 분할을 막는 안전장치이며, 여기 걸리면 일부가 누락됩니다.
- **최종 거리 필터** — 분할 원들이 원래 반경 밖까지 삐져나가므로 마지막에 걸러냅니다.
  또 카카오가 준 `distance` 는 **하위 셀 중심 기준**이라 쓸 수 없어, 원래 중심 기준으로 다시 계산합니다.
- **`visited`** — 탐색한 셀의 중심·반경·깊이를 기록해 둡니다. 수집에는 쓰이지 않고,
  8-5절에서 분할 과정을 그리는 데 씁니다. 스택에 `depth` 를 함께 넣어 몇 번째 분할인지 추적합니다.

In [ ]:
def collect_by_grid(query, lng, lat, radius, min_radius=50, **kwargs):
    """반경을 재귀적으로 4분할하며 45건 상한을 우회해 수집한다."""
    found, visited = {}, []
    stack = [(lng, lat, float(radius), 0)]    # 마지막 값은 분할 깊이
    give_up = 0

    while stack:
        cx, cy, r, depth = stack.pop()
        docs, truncated = fetch_cell(query, cx, cy, r, **kwargs)

        for d in docs:
            found[d["id"]] = d                # place id 로 중복 제거

        # 8-5절 시각화를 위해 탐색한 셀을 기록해 둔다
        visited.append({"lng": cx, "lat": cy, "radius": r, "depth": depth,
                        "found": len(docs), "truncated": truncated})

        if not truncated:
            continue
        if r <= min_radius:                   # 더 못 쪼갬 - 여기는 누락 가능
            give_up += 1
            continue

        half = r / 2
        sub_r = r * 0.7072                    # 사분면 정사각형을 덮는 최소 반경
        for sx in (-half, half):
            for sy in (-half, half):
                stack.append((*move(cx, cy, sx, sy), sub_r, depth + 1))

    # 분할 원이 원래 반경 밖까지 덮으므로 최종 거리 필터
    results = []
    for d in found.values():
        dist = distance_m(lng, lat, float(d["x"]), float(d["y"]))
        if dist <= radius:
            results.append(dict(d, distance=str(int(dist))))
    results.sort(key=lambda d: int(d["distance"]))

    return results, {"cells": len(visited), "give_up": give_up, "visited": visited}

### 8-4. 실행

**주의: 호출 횟수가 많습니다.** 가짜 데이터로 검증했을 때 1km 반경 하나에 **API 171회**가 나갔습니다
(단순 방식은 3회). 밀집 지역을 넓은 반경으로 돌리면 수천 회까지 늘어날 수 있으니,
처음에는 `radius` 를 작게 잡고 시작하세요.

`give_up` 이 0보다 크면 최소 반경까지 쪼갰는데도 상한에 걸린 곳이 있다는 뜻이라,
그만큼 결과가 누락됩니다. 그럴 땐 `min_radius` 를 더 줄여 보세요.

In [ ]:
all_places, stats = collect_by_grid("카페", center["lng"], center["lat"], 1000)

print(f"{len(all_places)}건 수집 (셀 {stats['cells']}개)")

if stats["give_up"]:
    print(f"경고: 상한에 걸린 셀 {stats['give_up']}개 - 일부 누락되었습니다.")

In [ ]:
# 단순 방식과 비교
print("단순 방식:", len(places), "건")
print("격자 분할:", len(all_places), "건")

In [ ]:
grid_df = pd.DataFrame(all_places)[
    ["place_name", "category_name", "road_address_name", "distance", "x", "y"]
].rename(columns={"x": "lng", "y": "lat"})
grid_df["distance"] = grid_df["distance"].astype(int)

grid_df.to_csv("kakao_places_grid.csv", index=False, encoding="utf-8-sig")
grid_df.head(10)

### 8-5. 분할 과정 시각화

탐색한 셀들을 **반경 크기대로 겹쳐 그립니다.** 분할이 밀도에 적응한다는 점이 한눈에 보입니다.

- 장소가 적은 구역은 **큰 원 하나**로 끝납니다 (한 번의 검색으로 충분).
- 장소가 밀집한 구역은 45건 상한에 계속 걸려 **작은 원이 촘촘하게** 쌓입니다.
- 원을 반투명하게 채웠기 때문에 **여러 번 쪼개진 곳일수록 색이 진해집니다.**
  결과적으로 상권 밀도 지도처럼 읽힙니다.

**코드 설명**

- `sorted(visited, key=lambda v: -v["radius"])` — **큰 원부터** 그립니다.
  작은 원을 먼저 그리면 나중에 그려진 큰 원에 덮여 보이지 않습니다.
- `alpha=0.10` — 겹침이 누적되어야 밀도가 드러나므로 낮게 잡습니다.
- 여기서도 반경 원은 `Circle` 이 아니라 `Ellipse` 입니다 (7-1절과 같은 이유).
- 깊이 `d` 의 반경은 `radius × 0.7072^d` 입니다. 범례에 실제 미터로 환산해 표시합니다.
- `bbox_to_anchor=(1.02, 1.0)` — 범례를 그래프 밖으로 빼 지도를 가리지 않게 합니다.

In [ ]:
from matplotlib.lines import Line2D


def plot_grid_cells(visited, center, radius, places=None, title="격자 분할 과정"):
    """탐색한 셀을 반경 크기대로 겹쳐 그린다. 여러 번 쪼개진 곳일수록 진해진다."""
    fig, ax = plt.subplots(figsize=(9, 8.5))

    depths = sorted({v["depth"] for v in visited})
    cmap = plt.get_cmap("plasma")
    color_of = {d: cmap(0.12 + 0.72 * (d / max(depths, default=1))) for d in depths}

    for v in sorted(visited, key=lambda v: -v["radius"]):      # 큰 원부터 깔아야 안 가려짐
        dl = v["radius"] / M_PER_DEG_LAT
        dg = v["radius"] / (M_PER_DEG_LAT * math.cos(math.radians(v["lat"])))
        ax.add_patch(Ellipse((v["lng"], v["lat"]), 2 * dg, 2 * dl,
                             facecolor=color_of[v["depth"]], alpha=0.10,
                             edgecolor=color_of[v["depth"]], lw=0.7, zorder=2))

    if places:
        ax.scatter([float(p["x"]) for p in places], [float(p["y"]) for p in places],
                   s=5, c="#222222", alpha=0.45, zorder=4,
                   label=f"수집 결과 ({len(places)})")

    ax.scatter([center["lng"]], [center["lat"]], marker="*", s=380, color="crimson",
               edgecolors="white", linewidths=1, zorder=6, label="중심")

    deg_lat = radius / M_PER_DEG_LAT
    deg_lng = radius / (M_PER_DEG_LAT * math.cos(math.radians(center["lat"])))
    ax.add_patch(Ellipse((center["lng"], center["lat"]), 2 * deg_lng, 2 * deg_lat,
                         fill=False, color="crimson", ls="--", lw=1.3, zorder=5))

    handles = [
        Line2D([], [], marker="o", ls="", markerfacecolor=color_of[d],
               markeredgecolor=color_of[d], alpha=0.75, markersize=9,
               label=f"깊이 {d} — 반경 {radius * (0.7072 ** d):.0f}m "
                     f"({sum(1 for v in visited if v['depth'] == d)}개)")
        for d in depths
    ]
    ax.legend(handles=handles + ax.get_legend_handles_labels()[0],
              loc="upper left", bbox_to_anchor=(1.02, 1.0),
              fontsize=9, framealpha=0.95, borderaxespad=0)

    ax.set_aspect(1 / math.cos(math.radians(center["lat"])))
    ax.set_xlim(center["lng"] - deg_lng * 1.2, center["lng"] + deg_lng * 1.2)
    ax.set_ylim(center["lat"] - deg_lat * 1.2, center["lat"] + deg_lat * 1.2)
    ax.set_title(f"{title} — 탐색 셀 {len(visited)}개")
    ax.set_xlabel("경도")
    ax.set_ylabel("위도")
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
fig, ax = plot_grid_cells(stats["visited"], center, 1000, places=all_places)
fig.savefig("kakao_grid_cells.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# 깊이별 셀 개수 - 어느 단계에서 호출이 많이 나갔는지
from collections import Counter

depth_counts = Counter(v["depth"] for v in stats["visited"])
for d in sorted(depth_counts):
    r = 1000 * (0.7072 ** d)
    n = depth_counts[d]
    print(f"깊이 {d}: 셀 {n:3d}개  (반경 {r:6.0f}m)  {'█' * min(n, 60)}")

print()
print("총 셀:", sum(depth_counts.values()))
print("상한에 걸린 셀:", sum(1 for v in stats["visited"] if v["truncated"]))

**비용을 줄이려면**

- `min_radius` 를 키우면 호출은 줄지만 누락이 늘어납니다 (트레이드오프).
- 같은 조건을 반복 호출하지 않도록 결과를 CSV/DB 에 캐싱하세요.
- 넓은 지역이 필요하면 키워드를 세분화하는 방법도 있습니다
  (`"카페"` 대신 `"스타벅스"`, `"투썸플레이스"` … 로 나눠 검색 후 합치기).

---
## 참고

| 항목 | 내용 |
| --- | --- |
| 좌표 순서 | `x` = 경도(lng), `y` = 위도(lat) |
| 결과 상한 | 키워드·카테고리 검색은 조건당 최대 **45건** (`pageable_count`) |
| `size` | 주소 검색 1~30, 키워드·카테고리 1~15 |
| `page` | 1~45 |
| `radius` | 0~20000 (m) |
| 응답 타입 | `x`, `y`, `distance` 는 모두 **문자열** — 계산 전 변환 필요 |
| 쿼터 초과 | HTTP `429` — 호출 간격을 두거나 결과를 캐싱 |
| 인증 실패 | HTTP `401` — REST API 키인지, 헤더가 `KakaoAK <키>` 형식인지 확인 |

**자주 겪는 문제**

- `NameError: center is not defined` → 2번 셀을 먼저 실행하세요.
- `TypeError: 'NoneType' object is not subscriptable` → `geocode()` 가 `None` 을 반환했습니다. 주소를 확인하세요.
- 함수가 `None` 을 반환 → `return` 을 빠뜨렸거나, 고친 뒤 **정의 셀을 다시 실행하지 않은** 경우입니다.
- Jupyter 에 붙여넣을 때 자동 들여쓰기로 실행 코드가 함수 안으로 들어가면 무한 재귀가 납니다.
  그래서 이 노트북은 **정의 셀과 실행 셀을 분리**해 두었습니다.

[카카오 로컬 API 문서](https://developers.kakao.com/docs/latest/ko/local/dev-guide)